# RAW to ACES Render Time Comparison

Measure and compare conversion times for converting RAW images to ACES format using rawtoaces.

## Import Required Libraries

Import necessary libraries for file handling, timing operations, and displaying results.

In [9]:
import os
import glob
import subprocess
import time
from pathlib import Path

## Load Sample Images from Dataset

Locate the first 10 RAW image files from `dataset/temp/raw` directory.

In [10]:
# Define the dataset path using absolute path
notebooks_dir = Path.cwd()
raw_dir = (notebooks_dir.parent / "dataset" / "temp" / "raw").resolve()

# Get all RAW files (CR2, NEF, ARW, RAF, DNG)
raw_extensions = ["*.CR2", "*.NEF", "*.ARW", "*.RAF", "*.DNG"]
raw_files = []
for ext in raw_extensions:
    raw_files.extend(sorted(raw_dir.glob(ext)))

# Select first 10 files
sample_files = raw_files[:10]

print(f"Raw directory: {raw_dir}")
print(f"Total RAW files found: {len(raw_files)}")
print(f"Selected {len(sample_files)} files for conversion:")
for i, f in enumerate(sample_files, 1):
    print(f"  {i}. {f.name}")

Raw directory: /run/media/mikkelkp/Projects/Projects/med8_project/LuminaScale/dataset/temp/raw
Total RAW files found: 1000
Selected 10 files for conversion:
  1. 0_0.CR2
  2. 0_1.CR2
  3. 0_2.CR2
  4. 0_3.CR2
  5. 0_4.CR2
  6. 0_5.CR2
  7. 0_6.CR2
  8. 1000_0.CR2
  9. 1000_1.CR2
  10. 1000_2.CR2


## Convert RAW Images to ACES Format

Convert each RAW image to ACES format using the rawtoaces command.

In [11]:
# Setup output directory using absolute path
notebook_dir = Path.cwd()
output_dir = notebook_dir.parent / "dataset" / "temp" / "aces_converted"
output_dir.mkdir(parents=True, exist_ok=True)

# Also create with absolute path for rawtoaces
output_dir_abs = output_dir.resolve()

# Storage for timing data
conversion_times = []
results = []

print(f"Notebook directory: {notebook_dir}")
print(f"Output directory: {output_dir_abs}\n")
print("Starting conversions...")
print("-" * 60)

Notebook directory: /run/media/mikkelkp/Projects/Projects/med8_project/LuminaScale/notebooks
Output directory: /run/media/mikkelkp/Projects/Projects/med8_project/LuminaScale/dataset/temp/aces_converted

Starting conversions...
------------------------------------------------------------


In [ ]:
# Convert each file and record timing, then display statistics
for idx, raw_file in enumerate(sample_files, 1):
    filename = raw_file.name
    output_path = output_dir_abs / f"{raw_file.stem}.exr"
    
    # Measure conversion time
    start_time = time.time()
    
    try:
        # Run rawtoaces command with proper flags (matching bash script)
        cmd = [
            "rawtoaces",
            "--data-dir", "/usr/local/share/rawtoaces/data",
            "--output-dir", str(output_dir_abs),
            "--create-dirs",
            "--overwrite",
            str(raw_file.resolve())
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        
        elapsed_time = time.time() - start_time
        
        if result.returncode == 0:
            status = "Success"
        else:
            status = "Failed"
        
        conversion_times.append(elapsed_time)
        results.append({
            "Image": filename,
            "Time (seconds)": elapsed_time,
            "Status": status,
            "Output": output_path.name
        })
        
    except subprocess.TimeoutExpired:
        conversion_times.append(None)
        results.append({
            "Image": filename,
            "Time (seconds)": "Timeout",
            "Status": "Timeout",
            "Output": None
        })
    except Exception as e:
        conversion_times.append(None)
        results.append({
            "Image": filename,
            "Time (seconds)": "Error",
            "Status": "Error",
            "Output": None
        })

# Calculate and display statistics for successful conversions
valid_times = [t for t in conversion_times if isinstance(t, (int, float))]

if valid_times:
    total_time = sum(valid_times)
    avg_time = total_time / len(valid_times)
    min_time = min(valid_times)
    max_time = max(valid_times)
    
    print(f"\nConversion Time Statistics:")
    print(f"  Total time: {total_time:.2f} seconds")
    print(f"  Average time per image: {avg_time:.2f} seconds")
    print(f"  Minimum time: {min_time:.2f} seconds")
    print(f"  Maximum time: {max_time:.2f} seconds")
    print(f"  Successfully converted: {len(valid_times)}/{len(sample_files)}")
else:
    print("\nNo successful conversions to report.")

1. Converting: 0_0.CR2
   ✓ Completed in 1.20 seconds
2. Converting: 0_1.CR2
   ✓ Completed in 1.23 seconds
3. Converting: 0_2.CR2
   ✓ Completed in 1.19 seconds
4. Converting: 0_3.CR2
   ✓ Completed in 1.20 seconds
5. Converting: 0_4.CR2
   ✓ Completed in 1.21 seconds
6. Converting: 0_5.CR2
   ✓ Completed in 1.18 seconds
7. Converting: 0_6.CR2
   ✓ Completed in 1.14 seconds
8. Converting: 1000_0.CR2
   ✓ Completed in 1.01 seconds
9. Converting: 1000_1.CR2
   ✓ Completed in 1.07 seconds
10. Converting: 1000_2.CR2
   ✓ Completed in 1.10 seconds
------------------------------------------------------------
